# Serve Current Facts with a Temporal Memory Store

## What you will build

You will build the memory for a support assistant at a software company. The assistant remembers facts about each customer account: the plan, the renewal owner and the support tier. Those facts change. An assistant that only adds new notes to the old ones keeps serving the old fact, long after it stopped being true.

The diagram shows that problem. The notes pile up, the old answer stays on top, and the customer gets an email from a person who has already left the company.

![What you will build](images/memory-desk-overview.svg)

## Step 0: Write the account facts and the dated questions

We write the facts and the questions as plain data, so every later step reads the same rows. Each fact carries the day it started and the day it stopped being true.

In [1]:
from datetime import date

TODAY = "2026-11-13"
FACTS = [
    {"account": "Northwind", "field": "plan", "value": "Starter", "valid_from": "2025-03-01", "valid_to": "2026-06-01"},
    {"account": "Northwind", "field": "plan", "value": "Growth", "valid_from": "2026-06-01", "valid_to": None},
    {"account": "Northwind", "field": "renewal_owner", "value": "Dana Reyes", "valid_from": "2025-03-01", "valid_to": "2026-04-15"},
    {"account": "Northwind", "field": "renewal_owner", "value": "Marcus Lee", "valid_from": "2026-04-15", "valid_to": None},
    {"account": "Contoso Freight", "field": "plan", "value": "Growth", "valid_from": "2025-07-01", "valid_to": None},
    {"account": "Contoso Freight", "field": "support_tier", "value": "Basic", "valid_from": "2025-07-01", "valid_to": "2026-01-20"},
    {"account": "Contoso Freight", "field": "support_tier", "value": "Standard", "valid_from": "2026-01-20", "valid_to": "2026-09-01"},
    {"account": "Contoso Freight", "field": "support_tier", "value": "Premium", "valid_from": "2026-09-01", "valid_to": None},
    {"account": "Fabrikam Health", "field": "plan", "value": "Team", "valid_from": "2025-10-01", "valid_to": "2026-08-15"},
    {"account": "Fabrikam Health", "field": "plan", "value": "Enterprise", "valid_from": "2026-08-15", "valid_to": None},
    {"account": "Fabrikam Health", "field": "renewal_owner", "value": "Priya Nair", "valid_from": "2025-10-01", "valid_to": None},
]

print(f"{len(FACTS)} facts across {len({row['account'] for row in FACTS})} accounts")

11 facts across 3 accounts


Next come the twelve dated questions. Each one names an account, a field, a day and the answer that was true on that day. A `None` expected answer means no fact existed yet, and the store must say so rather than guess.

In [2]:
QUESTIONS = [
    {"account": "Northwind", "field": "plan", "as_of": "2026-06-01", "expected": "Growth"},
    {"account": "Northwind", "field": "renewal_owner", "as_of": "2026-04-15", "expected": "Marcus Lee"},
    {"account": "Northwind", "field": "plan", "as_of": "2025-01-01", "expected": None},
    {"account": "Fabrikam Health", "field": "plan", "as_of": "2025-09-30", "expected": None},
    {"account": "Fabrikam Health", "field": "plan", "as_of": TODAY, "expected": "Enterprise"},
    {"account": "Contoso Freight", "field": "support_tier", "as_of": TODAY, "expected": "Premium"},
    {"account": "Northwind", "field": "plan", "as_of": "2026-05-31", "expected": "Starter"},
    {"account": "Northwind", "field": "renewal_owner", "as_of": "2026-03-10", "expected": "Dana Reyes"},
    {"account": "Contoso Freight", "field": "support_tier", "as_of": "2025-12-31", "expected": "Basic"},
    {"account": "Contoso Freight", "field": "support_tier", "as_of": "2026-03-01", "expected": "Standard"},
    {"account": "Fabrikam Health", "field": "plan", "as_of": "2026-03-01", "expected": "Team"},
    {"account": "Contoso Freight", "field": "plan", "as_of": "2026-05-20", "expected": "Growth"},
]

print(f"{len(QUESTIONS)} dated questions, {sum(q['expected'] is None for q in QUESTIONS)} expect no answer")

12 dated questions, 2 expect no answer


## Step 1: Store facts the simple way and watch one go stale

We start with the simplest store, a list that only grows. Each new fact is appended, and a question takes the first row that matches the account and the field.

In [3]:
NAIVE_STORE = []


def remember_fact(store, account, field, value):
    """Append the fact to the list. Nothing is ever closed or removed."""
    store.append({"account": account, "field": field, "value": value})


def recall_fact(store, account, field):
    """Return the first value ever stored for this account and field."""
    for row in store:
        if row["account"] == account and row["field"] == field:
            return row["value"]
    return None


for fact in FACTS:
    remember_fact(NAIVE_STORE, fact["account"], fact["field"], fact["value"])
print(f"stored {len(NAIVE_STORE)} rows, and every one is still in the list")

stored 11 rows, and every one is still in the list


The function `answer_from_naive_store` ignores the day completely. The function `count_stale_answers` takes any answer function and counts the questions it gets wrong.

In [4]:
def answer_from_naive_store(account, field, as_of):
    """The naive store ignores the day, so it gives the same answer on every day."""
    return recall_fact(NAIVE_STORE, account, field)


def count_stale_answers(answer_function):
    """Count the questions where the answer differs from what was true on that day."""
    wrong = [q for q in QUESTIONS
             if answer_function(q["account"], q["field"], q["as_of"]) != q["expected"]]
    return len(wrong)


stale_naive = count_stale_answers(answer_from_naive_store)
print(f"stale_answers (naive store): {stale_naive} of {len(QUESTIONS)}")

stale_answers (naive store): 7 of 12


The naive store served a stale answer on 7 of the 12 questions. On the April 15 question it still named Dana Reyes as the renewal owner, although Marcus Lee had taken over that day.

## Step 2: Give each fact a valid-from and a valid-to date

A row in the naive store records when we wrote the note down, not when the fact was true. We need the second kind of date, so each row gets two more fields.

The store has two kinds of date. The day we wrote a row is one kind. The days a fact was true for the account are the other kind, and we call them **valid time**. A row holds its valid time as a start and an end.

In [5]:
def make_fact(account, field, value, valid_from, valid_to=None):
    """Build one fact with its valid time. An end on or before the start is refused."""
    start = date.fromisoformat(valid_from)
    end = None if valid_to is None else date.fromisoformat(valid_to)
    if end is not None and end <= start:
        raise ValueError(f"{account} {field} ends {valid_to}, on or before its start {valid_from}")
    return {"account": account, "field": field, "value": value,
            "valid_from": start, "valid_to": end}


TEMPORAL_STORE = [make_fact(**fact) for fact in FACTS]
print(f"built {len(TEMPORAL_STORE)} dated facts")
print(TEMPORAL_STORE[1])

built 11 dated facts
{'account': 'Northwind', 'field': 'plan', 'value': 'Growth', 'valid_from': datetime.date(2026, 6, 1), 'valid_to': None}


## Step 3: Close the old fact when a new one arrives

When Northwind moves to the Growth plan, the Starter row has to stop on that same day. We close it by setting its end date, and we never delete it, because the old plan was true for every day before.

![Close the old fact when a new one arrives](images/fact-timeline-step-1.svg)

In [6]:
def update_fact(store, account, field, value, changed_on):
    """Close the live fact for this field on the change day, then add the new one."""
    change_day = date.fromisoformat(changed_on)
    live = [row for row in store
            if row["account"] == account and row["field"] == field and row["valid_to"] is None]
    for row in live:
        if change_day <= row["valid_from"]:
            raise ValueError(f"{account} {field} cannot change on {changed_on}, before it started")
        row["valid_to"] = change_day
    new_fact = make_fact(account, field, value, changed_on)
    store.append(new_fact)
    return new_fact


def print_timeline(store, account):
    """One line per fact the account has held, oldest first within each field."""
    rows = sorted((r for r in store if r["account"] == account),
                  key=lambda r: (r["field"], r["valid_from"]))
    for row in rows:
        end = row["valid_to"] or "open"
        print(f"{row['field']:15} {row['valid_from']} to {end}: {row['value']}")

Now we run the change on a small store that holds only the Starter plan. The Growth plan starts on June 1.

In [7]:
demo_store = [make_fact("Northwind", "plan", "Starter", "2025-03-01")]
update_fact(demo_store, "Northwind", "plan", "Growth", "2026-06-01")
print_timeline(demo_store, "Northwind")
print(f"rows kept: {len(demo_store)}, so the old plan is still on record")

plan            2025-03-01 to 2026-06-01: Starter
plan            2026-06-01 to open: Growth
rows kept: 2, so the old plan is still on record


The full timeline for Northwind comes from the store that holds every fact. Each field keeps its whole history.

In [8]:
print_timeline(TEMPORAL_STORE, "Northwind")

plan            2025-03-01 to 2026-06-01: Starter
plan            2026-06-01 to open: Growth
renewal_owner   2025-03-01 to 2026-04-15: Dana Reyes
renewal_owner   2026-04-15 to open: Marcus Lee


## Step 4: Answer what was true on a given date

A question now carries a day, and the store answers with the row whose dates cover that day. The range is half-open, so the start day counts and the end day does not.

![Answer what was true on a given date](images/fact-timeline-step-2.svg)

A **half-open range** includes its start day and excludes its end day. So a change that lands on June 1 belongs to the new fact, and the old fact answers only up to the day before.

In [9]:
def get_fact_as_of(store, account, field, day):
    """Return the value true on this day, or None when no fact covers it."""
    for row in store:
        if row["account"] != account or row["field"] != field:
            continue
        ends_after = row["valid_to"] is None or day < row["valid_to"]
        if row["valid_from"] <= day and ends_after:
            return row["value"]
    return None


for text in ["2025-01-01", "2026-05-31", "2026-06-01"]:
    answer = get_fact_as_of(TEMPORAL_STORE, "Northwind", "plan", date.fromisoformat(text))
    print(f"{text}: {answer}")

2025-01-01: None
2026-05-31: Starter
2026-06-01: Growth


## Step 5: Serve only current facts to the agent

The agent should only ever see facts that were true on the day it is answering about. build_memory_context collects those facts for one account and returns the lines the prompt would carry.

![Serve only current facts to the agent](images/fact-timeline-step-3.svg)

In [10]:
def build_memory_context(store, account, day):
    """The lines the prompt carries: every field that is true on this day."""
    lines = []
    for field in sorted({row["field"] for row in store if row["account"] == account}):
        value = get_fact_as_of(store, account, field, day)
        if value is not None:
            lines.append(f"{field}: {value}")
    return lines


def answer_from_temporal_store(account, field, as_of):
    """Answer from the dated store, using the day the question asks about."""
    return get_fact_as_of(TEMPORAL_STORE, account, field, date.fromisoformat(as_of))

Next we print the context for Northwind on the day before the renewal owner changed, and on the day it changed.

In [11]:
for text in ["2026-04-14", "2026-04-15"]:
    print(f"Northwind context on {text}:")
    for line in build_memory_context(TEMPORAL_STORE, "Northwind", date.fromisoformat(text)):
        print(f"  {line}")

Northwind context on 2026-04-14:
  plan: Starter
  renewal_owner: Dana Reyes
Northwind context on 2026-04-15:
  plan: Starter
  renewal_owner: Marcus Lee


Now the same twelve questions go to the dated store. Each answer comes from the row whose dates cover the question's day.

In [12]:
stale_temporal = count_stale_answers(answer_from_temporal_store)
print(f"stale_answers (temporal store): {stale_temporal} of {len(QUESTIONS)}")

stale_answers (temporal store): 0 of 12


The dated store served a stale answer on 0 of the 12 questions. Each answer came from the one row whose dates covered that day.

## Step 6: Reject two facts that overlap in time

A store that accepts two facts for one field on the same day gives two answers for that day. add_fact_checked refuses any new fact whose days clash with a stored fact for the same field.

![Reject two facts that overlap in time](images/fact-timeline-step-4.svg)

In [13]:
def find_overlapping_fact(store, new_fact):
    """The first stored row whose days share at least one day with the new fact, or None."""
    for row in store:
        if row["account"] != new_fact["account"] or row["field"] != new_fact["field"]:
            continue
        row_end = row["valid_to"] or date.max
        new_end = new_fact["valid_to"] or date.max
        if new_fact["valid_from"] < row_end and row["valid_from"] < new_end:
            return row
    return None


def add_fact_checked(store, new_fact):
    """Add the fact only when no stored fact for the same field covers any of its days."""
    clash = find_overlapping_fact(store, new_fact)
    if clash is not None:
        raise ValueError(f"{new_fact['account']} {new_fact['field']} from {new_fact['valid_from']} "
                         f"overlaps {clash['value']} from {clash['valid_from']}")
    store.append(new_fact)
    return new_fact

Here we try to add a Scale plan from July 1. The Growth plan is still open that day, so the store must refuse it.

In [14]:
scale_plan = make_fact("Northwind", "plan", "Scale", "2026-07-01")
try:
    add_fact_checked(TEMPORAL_STORE, scale_plan)
except ValueError as error:
    print(f"rejected: {error}")
print(f"rows after the rejected add: {len(TEMPORAL_STORE)}")

rejected: Northwind plan from 2026-07-01 overlaps Growth from 2026-06-01
rows after the rejected add: 11


The store refused the add and still holds 11 rows, so the Growth plan keeps every day from June 1 onward.

## Step 7: Test the memory store without calling the model

Each rule above gets a test that runs with no key and no model call. A test names the rule it guards, so a broken rule fails with a message the reader can act on.

In [15]:
def test_change_day_belongs_to_new_fact():
    assert get_fact_as_of(TEMPORAL_STORE, "Northwind", "plan", date(2026, 5, 31)) == "Starter"
    assert get_fact_as_of(TEMPORAL_STORE, "Northwind", "plan", date(2026, 6, 1)) == "Growth"


def test_no_fact_before_the_first_day():
    assert get_fact_as_of(TEMPORAL_STORE, "Northwind", "plan", date(2025, 1, 1)) is None


def test_overlapping_fact_is_rejected():
    try:
        add_fact_checked(list(TEMPORAL_STORE), make_fact("Northwind", "plan", "Scale", "2026-07-01"))
    except ValueError:
        return
    raise AssertionError("an overlapping fact was accepted")

The next tests cover the old row, the end date that must come after the start, and the stale answer count.

In [16]:
def test_update_keeps_the_old_row():
    store = [make_fact("Northwind", "plan", "Starter", "2025-03-01")]
    update_fact(store, "Northwind", "plan", "Growth", "2026-06-01")
    assert len(store) == 2, "the old row was deleted"
    assert store[0]["valid_to"] == date(2026, 6, 1)


def test_end_before_start_is_refused():
    try:
        make_fact("Northwind", "plan", "Starter", "2026-06-01", "2026-05-01")
    except ValueError:
        return
    raise AssertionError("a fact that ends before it starts was accepted")


def test_temporal_store_serves_no_stale_answers():
    assert count_stale_answers(answer_from_temporal_store) == 0


def test_naive_store_serves_stale_answers():
    assert count_stale_answers(answer_from_naive_store) > 0

Each test runs in one call. If any rule breaks, the cell stops on the failing assertion.

In [17]:
for test in (test_change_day_belongs_to_new_fact, test_no_fact_before_the_first_day,
             test_overlapping_fact_is_rejected, test_update_keeps_the_old_row,
             test_end_before_start_is_refused, test_temporal_store_serves_no_stale_answers,
             test_naive_store_serves_stale_answers):
    test()
    print(f"passed: {test.__name__}")

passed: test_change_day_belongs_to_new_fact
passed: test_no_fact_before_the_first_day
passed: test_overlapping_fact_is_rejected
passed: test_update_keeps_the_old_row
passed: test_end_before_start_is_refused
passed: test_temporal_store_serves_no_stale_answers
passed: test_naive_store_serves_stale_answers


## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Valid time** | `make_fact` | Stores the days a fact was true, separate from the day the row was written |
| **Close, never delete** | `update_fact` | Ends the old fact on the day the new one starts and keeps its row |
| **As-of lookup** | `get_fact_as_of` | Answers what was true on a given day, or `None` when nothing covers it |
| **Half-open range** | `get_fact_as_of` | Counts the start day and not the end day, so a change day belongs to the new fact |
| **Overlap check** | `find_overlapping_fact` and `add_fact_checked` | Refuses a fact whose days clash with a stored fact for the same field |
| **Context for the prompt** | `build_memory_context` | Lists only the facts true on the day the question asks about |